In [4]:
import gymnasium as gym
from gymnasium.wrappers import TimeLimit, RecordVideo
import numpy as np
import mujoco
env = gym.make('Pusher-v5', render_mode='rgb_array')
env = TimeLimit(env.env, max_episode_steps=200)
env = RecordVideo(env, "videos")
obs, info = env.reset()

print(env.spec.max_episode_steps)

200


In [5]:
Kp_pos = 0.6
Kp_rot = 0.075

damping = 0.1    # 阻尼系数（避免奇异）
max_steps = 2000

for i in range(max_steps):
    data = env.unwrapped.data
    model = env.unwrapped.model
    
    # 获取ee的信息
    ee_body_id = model.body("tips_arm").id    #10
    ee_pos = data.xpos[ee_body_id].copy()
    ee_rot = data.xmat[ee_body_id].reshape(3,3)
    # print ("ee_body_id: ", ee_body_id)
    # print ("ee_pos: ", ee_pos)
    # print ("ee_rot: ", ee_rot)
    
    # 获取obj的信息
    object_body_id = model.body("object").id
    object_pos = data.xpos[object_body_id].copy()
    # print ("object_body_id: ", object_body_id)
    # print ("object_pos: ", object_pos)
    
    # 获取goal位置
    goal_body_id = model.body("goal").id
    goal_pos = data.xpos[goal_body_id].copy()
    # print ("goal_body_id: ", goal_body_id)
    # print ("goal_pos: ", goal_pos)
    
    # 计算物体到目标的方向
    push_dir = goal_pos - object_pos
    norm = np.linalg.norm(push_dir)
    if norm > 1e-6:
        push_dir = push_dir / norm
    # print ("push_dir: ", push_dir)
    
    # 机械臂需要先移动到物体后面
    push_point = object_pos - 0.1 * push_dir
    
    """2. 计算error"""
    pos_error = object_pos - ee_pos
    # 夹爪向下， z轴朝下
    desired_rot = np.array([
        [-1,0,0],
        [0,-1,0],
        [0,0,1]
    ])
    # 姿态误差
    rot_error = 0.5 * (
        np.cross(ee_rot[:,0], desired_rot[:,0]) +
        np.cross(ee_rot[:,1], desired_rot[:,1]) +
        np.cross(ee_rot[:,2], desired_rot[:,2])
    )
    
    
    """3. pid算法求v_pos and v_rot， 合成desired_velocity"""
    v_pos = Kp_pos * pos_error
    v_rot = Kp_rot * rot_error
    desired_velocity = np.concatenate([v_pos, v_rot])
    
    
    """4. 计算 Jacobian, 将关节速度映射到末端速度"""
    # v = J(q) * q_dot， 这里 v 为 desired_velocity, q_dot是关节速度，我们需要反过来求 q_dot。
    # J 通常拆成两部分：Jp 线速度 Jacobian, Jr 角速度 Jacobian
    # 这里model.nv=11， 机械臂 7 + 物体 3 + 目标 1， 共11DOF
    jacp = np.zeros((3, model.nv))    # 末端线速度 Jacobian, v_linear = jacp * q̇
    jacr = np.zeros((3, model.nv))    # 末端角速度 Jacobian, ω = jacr * q̇
    # print("nq:", model.nq)    # 广义坐标（position）维度 = 11
    # print("nv:", model.nv)    # 广义速度（velocity）维度 = 11
    # print("nu:", model.nu)    # actuator（控制输入）数量 = 7
    # print ("jacp: ", jacp)
    
    #这是 MuJoCo 的 API，用来计算 Jacobian
    #jacp = ∂x / ∂q， jacr = ∂θ / ∂q
    #jacp =
    # [ dx/dq1 dx/dq2 ... ]
    # [ dy/dq1 dy/dq2 ... ]
    # [ dz/dq1 dz/dq2 ... ]
    # print (jacp)
    mujoco.mj_jacBody(
            model,
            data,
            jacp,    # 输出
            jacr,    # 输出
            ee_body_id
        )

    # Damped Least Squares IK： J⁺ = Jᵀ (J Jᵀ + λ²I)⁻¹
    J = np.vstack((jacp, jacr)) # 拼接
    JJt = J @ J.T
    I = np.eye(6)
    inv = np.linalg.inv(JJt + damping**2 * I)
    J_pseudo = J.T @ inv
    
    #逆运动学， 计算 q_dot = J^T * v
    # joint_velocity = jacp.T @ desired_velocity
    joint_velocity = J_pseudo @ desired_velocity
    
    # actuator 维度 = 7
    action_dim = env.action_space.shape[0]
    action = joint_velocity[:action_dim]
    # 控制输入范围
    action = np.clip(action, -0.6, 0.6)
    
    """5. 执行动作"""
    obs, reward, terminated, truncated, info = env.step(action)
    if terminated or truncated:
            obs, info = env.reset()
        
env.close()